
# 00 — Audit Data Penelitian

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 150)

# ============================================================
# PROJECT PATH
# ============================================================

CURRENT_DIR = Path.cwd()

# Notebook bisa dijalankan dari root project atau folder simulasi
if CURRENT_DIR.name == "simulasi":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

ORIGINAL_DIR = PROJECT_ROOT / "dataset" / "original"
PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "output" / "00_audit"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Original dir :", ORIGINAL_DIR)
print("Processed dir:", PROCESSED_DIR)
print("Output dir   :", OUTPUT_DIR)

Project root : /Users/riskyduha/Documents/Tesis/Prediction_Non_Adherent
Original dir : /Users/riskyduha/Documents/Tesis/Prediction_Non_Adherent/dataset/original
Processed dir: /Users/riskyduha/Documents/Tesis/Prediction_Non_Adherent/dataset/processed
Output dir   : /Users/riskyduha/Documents/Tesis/Prediction_Non_Adherent/output/00_audit


In [2]:
csv_files = list(ORIGINAL_DIR.glob("*Medication Adherence Final*.csv"))

if len(csv_files) == 0:
    raise FileNotFoundError(
        f"File CSV tidak ditemukan pada folder: {ORIGINAL_DIR}"
    )

if len(csv_files) > 1:
    print("Ditemukan lebih dari satu CSV:")
    for f in csv_files:
        print("-", f.name)

DATA_PATH = csv_files[0]

print("Dataset yang digunakan:")
print(DATA_PATH)

Dataset yang digunakan:
/Users/riskyduha/Documents/Tesis/Prediction_Non_Adherent/dataset/original/Cimas NCD Patient and  Medical Claims Data Medication Adherence Final.csv


In [3]:
df_raw = pd.read_csv(DATA_PATH)

print("Ukuran dataset saat pertama dibaca:")
print(f"Baris  : {df_raw.shape[0]:,}")
print(f"Kolom  : {df_raw.shape[1]}")

display(df_raw.head())

Ukuran dataset saat pertama dibaca:
Baris  : 8,142
Kolom  : 16


,BENEFICIARY,YEAR,Diagnosis,AGE,AGEGROUP,SCHEMETYPE,HOSPITALCOVERTYPE,ANNUALCONTRIBUTION,WELLNESSPROGRAM,GENDER,ANNUALCLAIMAMOUNT,FINALTHRESHOLD,COMPLICATIONDVELOPMENT,COMORBIDITY,COMORBIDITY.1,UNITS
0,3260976114,2022,Diabetes,16,16-20,MEDIUMLEVEL,PRIVATE_GRADEA,900,SUBSCRIBED,MALE,219.65,ADHERENT,YES,NO,NO,69
1,3958374122,2022,Diabetes,16,16-20,MEDIUMLEVEL,PRIVATE_GRADEA,900,SUBSCRIBED,FEMALE,928.35,ADHERENT,YES,NO,NO,397
2,3130974113,2022,Diabetes,17,16-20,MEDIUMLEVEL,PRIVATE_GRADEA,900,SUBSCRIBED,FEMALE,581.64,NON-ADHERENT,YES,NO,NO,418
3,3278475122,2022,Diabetes,17,16-20,MEDIUMLEVEL,PRIVATE_GRADEA,900,SUBSCRIBED,FEMALE,328.61,NON-ADHERENT,NO,NO,NO,21
4,3395816142,2022,Diabetes,17,16-20,MEDIUMLEVEL,BASIC_TO_GRADEB,768,NOT_SUBSCRIBED,FEMALE,210.93,NON-ADHERENT,YES,YES,YES,42


In [4]:
column_summary = pd.DataFrame({
    "no": range(1, len(df_raw.columns) + 1),
    "atribut": df_raw.columns
})

display(column_summary)

,no,atribut
0,1,BENEFICIARY
1,2,YEAR
2,3,Diagnosis
3,4,AGE
4,5,AGEGROUP
5,6,SCHEMETYPE
6,7,HOSPITALCOVERTYPE
7,8,ANNUALCONTRIBUTION
8,9,WELLNESSPROGRAM
9,10,GENDER


In [5]:
repeated_header_mask = (
    df_raw["BENEFICIARY"]
    .astype(str)
    .str.strip()
    .eq("BENEFICIARY")
)

repeated_header_rows = df_raw.loc[repeated_header_mask]

print("Jumlah baris header berulang:", repeated_header_mask.sum())

if len(repeated_header_rows) > 0:
    display(repeated_header_rows)

Jumlah baris header berulang: 1


,BENEFICIARY,YEAR,Diagnosis,AGE,AGEGROUP,SCHEMETYPE,HOSPITALCOVERTYPE,ANNUALCONTRIBUTION,WELLNESSPROGRAM,GENDER,ANNUALCLAIMAMOUNT,FINALTHRESHOLD,COMPLICATIONDVELOPMENT,COMORBIDITY,COMORBIDITY.1,UNITS
1680,BENEFICIARY,YEAR,Diagnosis,AGE,AGEGROUP,SCHEMETYPE,HOSPITALCOVERTYPE,ANNUALCONTRIBUTION,WELLNESSPROGRAM,GENDER,ANNUALCLAIMAMOUNT,FINALTHRESHOLD,COMPLICATIONDVELOPMENT,COMORBIDITY,COMORBIDITY,UNITS


In [6]:
df = df_raw.loc[~repeated_header_mask].copy()

df.reset_index(drop=True, inplace=True)

print("Jumlah observasi setelah menghapus header berulang:")
print(f"{len(df):,}")

Jumlah observasi setelah menghapus header berulang:
8,141


In [7]:
numeric_columns_raw = [
    "BENEFICIARY",
    "YEAR",
    "AGE",
    "ANNUALCONTRIBUTION",
    "ANNUALCLAIMAMOUNT",
    "UNITS"
]

for col in numeric_columns_raw:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.dtypes)

BENEFICIARY                 int64
YEAR                        int64
Diagnosis                     str
AGE                         int64
AGEGROUP                      str
SCHEMETYPE                    str
HOSPITALCOVERTYPE             str
ANNUALCONTRIBUTION          int64
WELLNESSPROGRAM               str
GENDER                        str
ANNUALCLAIMAMOUNT         float64
FINALTHRESHOLD                str
COMPLICATIONDVELOPMENT        str
COMORBIDITY                   str
COMORBIDITY.1                 str
UNITS                       int64
dtype: object


In [8]:
full_duplicate_count = int(df.duplicated().sum())

print("Jumlah duplikasi observasi penuh:", full_duplicate_count)

if full_duplicate_count > 0:
    display(df[df.duplicated(keep=False)].sort_values(list(df.columns)))

Jumlah duplikasi observasi penuh: 0


In [9]:
missing_summary = (
    df.isna()
    .sum()
    .reset_index()
)

missing_summary.columns = ["atribut", "jumlah_missing"]

missing_summary["persentase_missing"] = (
    missing_summary["jumlah_missing"] / len(df) * 100
).round(4)

missing_summary = missing_summary.sort_values(
    "jumlah_missing",
    ascending=False
).reset_index(drop=True)

display(missing_summary)

missing_summary.to_csv(
    OUTPUT_DIR / "04_missing_values.csv",
    index=False
)

,atribut,jumlah_missing,persentase_missing
0,GENDER,5,0.0614
1,BENEFICIARY,0,0.0000
2,YEAR,0,0.0000
3,Diagnosis,0,0.0000
4,AGE,0,0.0000
5,AGEGROUP,0,0.0000
6,SCHEMETYPE,0,0.0000
7,HOSPITALCOVERTYPE,0,0.0000
8,ANNUALCONTRIBUTION,0,0.0000
9,WELLNESSPROGRAM,0,0.0000


In [10]:
categorical_columns_audit = [
    "Diagnosis",
    "AGEGROUP",
    "SCHEMETYPE",
    "HOSPITALCOVERTYPE",
    "WELLNESSPROGRAM",
    "GENDER",
    "FINALTHRESHOLD",
    "COMPLICATIONDVELOPMENT",
    "COMORBIDITY",
    "COMORBIDITY.1"
]

category_records = []

for col in categorical_columns_audit:
    counts = df[col].value_counts(dropna=False)

    for value, count in counts.items():
        category_records.append({
            "atribut": col,
            "kategori": value,
            "jumlah": int(count)
        })

category_audit = pd.DataFrame(category_records)

display(category_audit)

category_audit.to_csv(
    OUTPUT_DIR / "05_category_audit.csv",
    index=False
)

,atribut,kategori,jumlah
0,Diagnosis,Hypetension,6461
1,Diagnosis,Diabetes,1680
2,AGEGROUP,40-60,4271
3,AGEGROUP,60-80,2102
4,AGEGROUP,20-40,1434
5,AGEGROUP,81+,311
6,AGEGROUP,16-20,23
7,SCHEMETYPE,MEDIUMLEVEL,5180
8,SCHEMETYPE,PREMIUMLEVEL,1508
9,SCHEMETYPE,BASICLEVEL,1453


In [11]:
df["Diagnosis"] = df["Diagnosis"].replace({
    "Hypetension": "Hypertension"
})

df["HOSPITALCOVERTYPE"] = df["HOSPITALCOVERTYPE"].replace({
    "Basic_TO_GradeB": "BASIC_TO_GRADEB"
})

In [12]:
print("Diagnosis setelah standardisasi:")
display(df["Diagnosis"].value_counts())

print("\nHospital Cover Type setelah standardisasi:")
display(df["HOSPITALCOVERTYPE"].value_counts())

Diagnosis setelah standardisasi:


Diagnosis
Hypertension    6461
Diabetes        1680
Name: count, dtype: int64


Hospital Cover Type setelah standardisasi:


HOSPITALCOVERTYPE
BASIC_TO_GRADEB    4281
PRIVATE_GRADEA     3860
Name: count, dtype: int64

In [13]:
target_distribution = (
    df["FINALTHRESHOLD"]
    .value_counts()
    .rename_axis("kelas")
    .reset_index(name="jumlah")
)

target_distribution["persentase"] = (
    target_distribution["jumlah"] / len(df) * 100
).round(2)

display(target_distribution)

target_distribution.to_csv(
    OUTPUT_DIR / "02_target_distribution.csv",
    index=False
)

,kelas,jumlah,persentase
0,NON-ADHERENT,6600,81.07
1,ADHERENT,1541,18.93


In [14]:
n_nonadherent = int(
    (df["FINALTHRESHOLD"] == "NON-ADHERENT").sum()
)

n_adherent = int(
    (df["FINALTHRESHOLD"] == "ADHERENT").sum()
)

imbalance_ratio = n_nonadherent / n_adherent

print(f"Imbalance Ratio = {imbalance_ratio:.4f}:1")

Imbalance Ratio = 4.2829:1


In [15]:
beneficiary_obs_count = (
    df.groupby("BENEFICIARY")
    .size()
    .rename("jumlah_observasi")
)

n_unique_beneficiary = df["BENEFICIARY"].nunique()

n_single_obs = int(
    (beneficiary_obs_count == 1).sum()
)

n_multiple_obs = int(
    (beneficiary_obs_count > 1).sum()
)

max_obs_per_beneficiary = int(
    beneficiary_obs_count.max()
)

beneficiary_summary = pd.DataFrame({
    "indikator": [
        "Jumlah BENEFICIARY unik",
        "BENEFICIARY dengan 1 observasi",
        "BENEFICIARY dengan >1 observasi",
        "Maksimum observasi per BENEFICIARY"
    ],
    "nilai": [
        n_unique_beneficiary,
        n_single_obs,
        n_multiple_obs,
        max_obs_per_beneficiary
    ]
})

display(beneficiary_summary)

,indikator,nilai
0,Jumlah BENEFICIARY unik,6940
1,BENEFICIARY dengan 1 observasi,5739
2,BENEFICIARY dengan >1 observasi,1201
3,Maksimum observasi per BENEFICIARY,2


In [16]:
beneficiary_label_count = (
    df.groupby("BENEFICIARY")["FINALTHRESHOLD"]
    .nunique(dropna=True)
)

n_variable_label_beneficiary = int(
    (beneficiary_label_count > 1).sum()
)

print(
    "BENEFICIARY dengan lebih dari satu nilai FINALTHRESHOLD:",
    n_variable_label_beneficiary
)

BENEFICIARY dengan lebih dari satu nilai FINALTHRESHOLD: 113


In [17]:
variable_label_ids = beneficiary_label_count[
    beneficiary_label_count > 1
].index

beneficiary_label_detail = (
    df[df["BENEFICIARY"].isin(variable_label_ids)]
    [["BENEFICIARY", "FINALTHRESHOLD"]]
    .sort_values(["BENEFICIARY", "FINALTHRESHOLD"])
)

beneficiary_label_detail.to_csv(
    OUTPUT_DIR / "beneficiary_label_variation_detail.csv",
    index=False
)

In [18]:
beneficiary_audit = pd.DataFrame({
    "indikator": [
        "Jumlah BENEFICIARY unik",
        "BENEFICIARY dengan 1 observasi",
        "BENEFICIARY dengan >1 observasi",
        "Maksimum observasi per BENEFICIARY",
        "BENEFICIARY dengan label bervariasi"
    ],
    "nilai": [
        n_unique_beneficiary,
        n_single_obs,
        n_multiple_obs,
        max_obs_per_beneficiary,
        n_variable_label_beneficiary
    ]
})

beneficiary_audit.to_csv(
    OUTPUT_DIR / "03_beneficiary_audit.csv",
    index=False
)

display(beneficiary_audit)

,indikator,nilai
0,Jumlah BENEFICIARY unik,6940
1,BENEFICIARY dengan 1 observasi,5739
2,BENEFICIARY dengan >1 observasi,1201
3,Maksimum observasi per BENEFICIARY,2
4,BENEFICIARY dengan label bervariasi,113


In [19]:
comorbidity_crosstab = pd.crosstab(
    df["COMORBIDITY"],
    df["COMORBIDITY.1"],
    dropna=False
)

display(comorbidity_crosstab)

COMORBIDITY.1,NO,YES
COMORBIDITY,,
NO,479,0
YES,5260,2402


In [20]:
comorbidity_output = (
    comorbidity_crosstab
    .reset_index()
)

comorbidity_output.to_csv(
    OUTPUT_DIR / "06_comorbidity_audit.csv",
    index=False
)

In [21]:
feature_roles = pd.DataFrame([
    ["BENEFICIARY", "group_identifier", False,
     "Digunakan sebagai group untuk mencegah leakage antar-pasien"],

    ["YEAR", "excluded", False,
     "Seluruh observasi memiliki nilai 2022"],

    ["Diagnosis", "categorical_feature", True,
     "Fitur kategorikal"],

    ["AGE", "numeric_feature", True,
     "Fitur numerik"],

    ["AGEGROUP", "excluded", False,
     "Kategorisasi deterministik dari AGE"],

    ["SCHEMETYPE", "categorical_feature", True,
     "Fitur kategorikal"],

    ["HOSPITALCOVERTYPE", "categorical_feature", True,
     "Fitur kategorikal"],

    ["ANNUALCONTRIBUTION", "numeric_feature", True,
     "Fitur numerik"],

    ["WELLNESSPROGRAM", "categorical_feature", True,
     "Fitur kategorikal"],

    ["GENDER", "categorical_feature", True,
     "Fitur kategorikal; missing ditangani di dalam training fold"],

    ["ANNUALCLAIMAMOUNT", "numeric_feature", True,
     "Fitur numerik"],

    ["FINALTHRESHOLD", "target", False,
     "Variabel target klasifikasi"],

    ["COMPLICATIONDVELOPMENT", "categorical_feature", True,
     "Fitur kategorikal"],

    ["COMORBIDITY", "excluded", False,
     "Makna terpisah tidak dapat diverifikasi secara independen"],

    ["COMORBIDITY.1", "categorical_feature", True,
     "Dipertahankan sebagai status komorbiditas"],

    ["UNITS", "numeric_feature", True,
     "Fitur numerik"],
], columns=[
    "atribut",
    "peran",
    "digunakan_sebagai_prediktor",
    "alasan"
])

display(feature_roles)

feature_roles.to_csv(
    OUTPUT_DIR / "07_feature_roles.csv",
    index=False
)

,atribut,peran,digunakan_sebagai_prediktor,alasan
0,BENEFICIARY,group_identifier,False,Digunakan sebagai group untuk mencegah leakage...
1,YEAR,excluded,False,Seluruh observasi memiliki nilai 2022
2,Diagnosis,categorical_feature,True,Fitur kategorikal
3,AGE,numeric_feature,True,Fitur numerik
4,AGEGROUP,excluded,False,Kategorisasi deterministik dari AGE
5,SCHEMETYPE,categorical_feature,True,Fitur kategorikal
6,HOSPITALCOVERTYPE,categorical_feature,True,Fitur kategorikal
7,ANNUALCONTRIBUTION,numeric_feature,True,Fitur numerik
8,WELLNESSPROGRAM,categorical_feature,True,Fitur kategorikal
9,GENDER,categorical_feature,True,Fitur kategorikal; missing ditangani di dalam ...


In [22]:
dataset_summary = pd.DataFrame({
    "indikator": [
        "Observasi mentah terbaca",
        "Header berulang",
        "Observasi valid",
        "Jumlah atribut",
        "BENEFICIARY unik",
        "BENEFICIARY >1 observasi",
        "BENEFICIARY label bervariasi",
        "Duplikasi observasi penuh",
        "Missing value",
        "NON-ADHERENT",
        "ADHERENT",
        "Persentase NON-ADHERENT",
        "Persentase ADHERENT",
        "Imbalance Ratio"
    ],
    "nilai": [
        len(df_raw),
        int(repeated_header_mask.sum()),
        len(df),
        df.shape[1],
        n_unique_beneficiary,
        n_multiple_obs,
        n_variable_label_beneficiary,
        full_duplicate_count,
        int(df.isna().sum().sum()),
        n_nonadherent,
        n_adherent,
        round(n_nonadherent / len(df) * 100, 2),
        round(n_adherent / len(df) * 100, 2),
        round(imbalance_ratio, 4)
    ]
})

display(dataset_summary)

dataset_summary.to_csv(
    OUTPUT_DIR / "01_dataset_summary.csv",
    index=False
)

,indikator,nilai
0,Observasi mentah terbaca,8142.0000
1,Header berulang,1.0000
2,Observasi valid,8141.0000
3,Jumlah atribut,16.0000
4,BENEFICIARY unik,6940.0000
5,BENEFICIARY >1 observasi,1201.0000
6,BENEFICIARY label bervariasi,113.0000
7,Duplikasi observasi penuh,0.0000
8,Missing value,5.0000
9,NON-ADHERENT,6600.0000


In [23]:
audit_checks = {
    "observasi_valid_8141": len(df) == 8141,
    "jumlah_atribut_16": df.shape[1] == 16,
    "beneficiary_unik_6940": n_unique_beneficiary == 6940,
    "non_adherent_6600": n_nonadherent == 6600,
    "adherent_1541": n_adherent == 1541,
    "beneficiary_multi_1201": n_multiple_obs == 1201,
    "beneficiary_label_bervariasi_113": n_variable_label_beneficiary == 113,
    "tidak_ada_full_duplicate": full_duplicate_count == 0
}

audit_check_df = pd.DataFrame(
    audit_checks.items(),
    columns=["pemeriksaan", "lulus"]
)

display(audit_check_df)

audit_check_df.to_csv(
    OUTPUT_DIR / "08_audit_checks.csv",
    index=False
)

if audit_check_df["lulus"].all():
    print("✓ Seluruh audit check sesuai.")
else:
    print("PERINGATAN: terdapat audit check yang tidak sesuai.")

,pemeriksaan,lulus
0,observasi_valid_8141,True
1,jumlah_atribut_16,True
2,beneficiary_unik_6940,True
3,non_adherent_6600,True
4,adherent_1541,True
5,beneficiary_multi_1201,True
6,beneficiary_label_bervariasi_113,True
7,tidak_ada_full_duplicate,True


✓ Seluruh audit check sesuai.


In [24]:
PROCESSED_PATH = (
    PROCESSED_DIR /
    "cimas_medication_adherence_clean.csv"
)

df.to_csv(
    PROCESSED_PATH,
    index=False
)

print("Dataset bersih tersimpan:")
print(PROCESSED_PATH)

print("\nUkuran dataset:")
print(df.shape)

Dataset bersih tersimpan:
/Users/riskyduha/Documents/Tesis/Prediction_Non_Adherent/dataset/processed/cimas_medication_adherence_clean.csv

Ukuran dataset:
(8141, 16)
